In [1]:
# 1. Create the test Dask cluster. This will be used to auto-determine the appropriate block size
#    for your machine! 
from robustraster import dask_cluster_manager

json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="test")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [1]:
from robustraster import dask_cluster_manager

json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="full", threads_per_worker=1, n_workers=19, memory_limit="10GB")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [7]:
dask_client.shutdown()

In [2]:
# 2. Authenticate Google Earth Engine on all Dask workers.
from robustraster import dask_plugins

ee_plugin = dask_plugins.EEPlugin(json_key)
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)

{'tcp://127.0.0.1:61688': {'status': 'OK'},
 'tcp://127.0.0.1:61689': {'status': 'OK'},
 'tcp://127.0.0.1:61694': {'status': 'OK'},
 'tcp://127.0.0.1:61695': {'status': 'OK'},
 'tcp://127.0.0.1:61696': {'status': 'OK'},
 'tcp://127.0.0.1:61697': {'status': 'OK'},
 'tcp://127.0.0.1:61706': {'status': 'OK'},
 'tcp://127.0.0.1:61707': {'status': 'OK'},
 'tcp://127.0.0.1:61712': {'status': 'OK'},
 'tcp://127.0.0.1:61719': {'status': 'OK'},
 'tcp://127.0.0.1:61734': {'status': 'OK'},
 'tcp://127.0.0.1:61738': {'status': 'OK'},
 'tcp://127.0.0.1:61739': {'status': 'OK'},
 'tcp://127.0.0.1:61740': {'status': 'OK'},
 'tcp://127.0.0.1:61743': {'status': 'OK'},
 'tcp://127.0.0.1:61750': {'status': 'OK'},
 'tcp://127.0.0.1:61757': {'status': 'OK'},
 'tcp://127.0.0.1:61760': {'status': 'OK'},
 'tcp://127.0.0.1:61763': {'status': 'OK'}}

In [5]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json
import xarray as xr
from dask.distributed import performance_report
import dask

# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')


WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")


#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2014-01-01', '2014-12-31')
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2020-05-01', '2020-05-10').select(['SR_B4', 'SR_B5'])
xarray_data = xr.open_dataset(ic,
            engine='ee', 
            crs="EPSG:3310", 
            scale=30,
            geometry=WSDemo.geometry(),
            chunks={"time": 48, "X": 512, "Y": 256})

In [3]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json

# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')


WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
California = ee.FeatureCollection("projects/robust-raster/assets/boundaries/California")

# This is a dictionary that my code requires. You don't have to touch anything here for demo purposes
# (although it should work with anything, in theory. Feel free to change it if you'd like).
# These parameters get stored and are used to generate the header information needed when we do the full
# run.
parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-05-01',
            'end_date': '2020-05-10',
            'geometry': WSDemo.geometry(),
            'crs': 'EPSG:3310',
            'scale': 30
        }

earth_engine = dataset_manager.EarthEngineDataset(parameters)

In [4]:
print(earth_engine.dataset)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1, X: 459, Y: 546)
Coordinates:
  * time     (time) datetime64[ns] 8B 2020-05-05T18:44:42.065000
  * X        (X) float64 4kB -7.668e+04 -7.665e+04 ... -6.297e+04 -6.294e+04
  * Y        (Y) float64 4kB 1.886e+05 1.886e+05 ... 2.049e+05 2.05e+05
Data variables:
    SR_B4    (time, X, Y) float32 1MB ...
    SR_B5    (time, X, Y) float32 1MB ...
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,SR_B3
    visualization_2_max:    30000.0
    visualization_2_min:    0.0
    visualization_2_name:   

In [2]:
import xarray as xr
xarray_data = xr.open_dataset("./output2.nc", chunks={"time": 7, "X": 1024, "Y": 1024})

In [ ]:
print(xarray_data)

In [6]:
# 5. This code will auto-determine what the best block size
#    should be for your machine. This helps to ensure computations don't 
#    go over resources available and crash your application. Skip this step
#    if want to process the entire dataset as is.
from robustraster import udf_manager

def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

user_defined_func = udf_manager.UserDefinedFunction()
user_defined_func.tune_user_function(xarray_data, compute_ndvi, None)

Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
LATEST TPARALLEL IS GREATER THAN PREVIOUS TPARALLEL! STOPPING...
LATEST: 4.9629989935427295e-09
PREVIOUS: 4.4954066373864e-09
Optimal chunks written to optimal_chunks_20250224_145228.json


In [ ]:
import dask

@dask.delayed
def dataframe_conversion(ds):
    return ds.to_dataframe().reset_index()

@dask.delayed
def compute_ndvi(df):
    df["ndvi"] = (df["SR_B5"] - df["SR_B4"]) / (df["SR_B5"] + df["SR_B4"])
    return df

@dask.delayed
def dataframe_to_xarray(df, dims):
    df = df.set_index(dims)
    return df.to_xarray()

delayed_df = dataframe_conversion(xarray_data)
delayed_ndvi = compute_ndvi(delayed_df)
delayed_xr = dataframe_to_xarray(delayed_ndvi, list(xarray_data.dims))

# Compute everything at once
result = delayed_xr.compute()